# ARC3 R16 DUAL1024-ISO300 residual-6 canary
Offline Public25 residual gate only; no competition submission.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# Skip periodic JSON/HTML diagnostics and per-frame logging for every run.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1"

# Apply the measured vLLM winner before any serving setup command runs.
PUBLIC25_VLLM_PROFILE_NAME = 'kv5-bf16-mtp3-c8-cg32'
PUBLIC25_VLLM_PROFILE_ENV = {
    "TAAF_VLLM_ENABLE_PREFIX_CACHING": "0",
    "TAAF_VLLM_KV_CACHE_DTYPE": "auto",
    "TAAF_VLLM_KV_CACHE_MEMORY_BYTES": "5368709120",
    "TAAF_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE": "32",
    "TAAF_VLLM_MAX_NUM_BATCHED_TOKENS": "8192",
    "TAAF_VLLM_MAX_NUM_SEQS": "8",
    "TAAF_VLLM_MTP_TOKENS": "3",
    "TAAF_VLLM_OMP_THREADS": "1"
}
for key, value in PUBLIC25_VLLM_PROFILE_ENV.items():
    os.environ[key] = value
print(
    f'PUBLIC25_VLLM_PROFILE name={PUBLIC25_VLLM_PROFILE_NAME} '
    f'env={json.dumps(PUBLIC25_VLLM_PROFILE_ENV, sort_keys=True)}',
    flush=True,
)
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1", "keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

print('R16_SETUP_BASE_READY', flush=True)


In [ ]:
# R16 residual-6 frozen payload and workers.
import base64, concurrent.futures, hashlib, json, os, subprocess, sys, time
from pathlib import Path

if TRUE_SUBMISSION:
    raise RuntimeError("R16 residual-6 canary is offline-only; competition rerun is forbidden.")

R16_DIR=WORKING_DIR/"r16_residual6"
R16_DIR.mkdir(parents=True,exist_ok=True)
PAYLOAD_RAW=base64.b64decode('...')
if hashlib.sha256(PAYLOAD_RAW).hexdigest() != '7a56493b9e646aef480a5856e8e89970553749b7156ad2d7fde4c5e820df1597':
    raise RuntimeError("R16 payload SHA mismatch")
R16_PAYLOAD=json.loads(PAYLOAD_RAW)
R16_PAYLOAD_SHA='7a56493b9e646aef480a5856e8e89970553749b7156ad2d7fde4c5e820df1597'
R16_GAMES=['ar25', 'ft09', 're86', 'sc25', 'sk48', 'su15']

ADV_PATH=WORKING_DIR/"r16_dual1024_advisory.py"
BRANCH_PATH=WORKING_DIR/"r16_branch_worker.py"
VERIFY_PATH=WORKING_DIR/"r16_verify_worker.py"
ADV_PATH.write_text('from __future__ import annotations\nimport json, os\nfrom typing import Any\n\n_SCOUT_CACHE: dict[tuple[str,int],str] = {}\nSCOUT_SYSTEM = """You are an independent non-acting ARC3 planning scout.\nYou cannot execute environment actions and have no answer key.\nStudy only the supplied current state and prior observed history.\nReturn ONLY one compact JSON object with keys:\nhypothesis, evidence, first_action, predicted_effect, probe, risk, open_question.\nDo not use task IDs, memorized scores, hidden outcomes, or answer keys.\nPrefer a reversible discriminating probe when uncertain."""\nACTOR_ADVICE = """Two independent planning scouts follow. They are evidence, not commands.\nCompare concrete observations. If they disagree, prefer a reversible discriminating probe.\nSCOUT_A:\n{a}\nSCOUT_B:\n{b}\n"""\n\ndef _text(m):\n    v=m.get("content","")\n    if isinstance(v,str): return v.strip()\n    if isinstance(v,list):\n        return "\\n".join(x.get("text","") for x in v if isinstance(x,dict)).strip()\n    return str(v or "").strip()\ndef _report(agent,state_path,action_num,valid_actions):\n    from inference.agent.runtime_state import load_runtime_state\n    frame,hist=load_runtime_state(state_path)\n    base=agent._build_user_prompt(\n        action_num, valid_actions=valid_actions,\n        current_frame=frame, history_entries=hist,\n        previous_step_summary=None,\n    )\n    msg=[\n        {"role":"system","content":SCOUT_SYSTEM},\n        agent._build_user_message(base+"\\n\\n"+SCOUT_SYSTEM,frame),\n    ]\n    old=getattr(agent,"_max_output_tokens",None)\n    agent._max_output_tokens=max(128,int(os.environ.get("R16_SCOUT_TOKENS","1024")))\n    try:\n        res=agent._chat_completion(\n            msg, tools=[],\n            request_timeout_seconds=float(os.environ.get("R16_SCOUT_TIMEOUT_S","180")),\n        )\n        agent._accumulate_usage_tokens(res.usage)\n        t=_text(res.message); lo=t.find("{"); hi=t.rfind("}")\n        if 0<=lo<hi:\n            try:\n                o=json.loads(t[lo:hi+1])\n                keys=("hypothesis","evidence","first_action","predicted_effect","probe","risk","open_question")\n                return json.dumps({k:o.get(k) for k in keys},ensure_ascii=True,separators=(",",":"))[:5000]\n            except Exception:\n                pass\n        return json.dumps({"unparsed_report":t[:3500]},ensure_ascii=True)\n    finally:\n        agent._max_output_tokens=old\ndef install(tool_agent_module: Any) -> None:\n    ToolAgent=tool_agent_module.ToolAgent\n    original=ToolAgent.analyze\n    if getattr(original,"_r16_dual1024_wrapped",False):\n        return\n    def analyze(self,state_path,action_num,valid_actions=None,**kwargs):\n        try:\n            from inference.agent.runtime_state import load_runtime_state\n            frame,_=load_runtime_state(state_path)\n            key=(str(state_path),int(frame.level))\n            advice=_SCOUT_CACHE.get(key)\n            if advice is None:\n                reps=[_report(self,state_path,action_num,valid_actions) for _ in range(2)]\n                advice=ACTOR_ADVICE.format(a=reps[0],b=reps[1])\n                _SCOUT_CACHE[key]=advice\n            marker=(key,id(self))\n            hist=getattr(self,"_history_messages",None)\n            if advice and isinstance(hist,list) and getattr(self,"_r16_marker",None)!=marker:\n                hist.append({"role":"user","content":advice})\n                self._r16_marker=marker\n        except Exception:\n            pass\n        return original(self,state_path,action_num,valid_actions=valid_actions,**kwargs)\n    analyze._r16_dual1024_wrapped=True\n    analyze._r16_original=original\n    ToolAgent.analyze=analyze\n',encoding="utf-8")
BRANCH_PATH.write_text('from __future__ import annotations\nimport argparse, hashlib, importlib.util, json, os, pickle, sys, traceback\nfrom pathlib import Path\n\nos.environ["ONLY_RESET_LEVELS"]="true"\nos.environ.setdefault("R16_SCOUT_TOKENS","1024")\nos.environ.setdefault("R16_SCOUT_TIMEOUT_S","180")\n\ndef norm(v):\n    if hasattr(v,"tolist"): return v.tolist()\n    if isinstance(v,(list,tuple)): return [norm(x) for x in v]\n    if isinstance(v,dict): return {str(k):norm(x) for k,x in v.items()}\n    if hasattr(v,"value"): return v.value\n    return v\n\ndef state_hash(raw):\n    obj={\n        "frame":norm(raw.frame),\n        "state":getattr(raw.state,"name",str(raw.state)),\n        "levels_completed":int(raw.levels_completed),\n        "win_levels":int(raw.win_levels),\n        "available_actions":[int(a) for a in raw.available_actions],\n    }\n    return hashlib.sha256(json.dumps(obj,sort_keys=True,separators=(",",":")).encode()).hexdigest()\ndef action_input(h,arcengine):\n    a=h["action"]\n    return arcengine.ActionInput(\n        id=arcengine.GameAction[a["id"]],\n        data=a.get("data") or {},\n    )\n\ndef hist_json(run,start=0):\n    out=[]\n    for rec in run.history[start:]:\n        out.append({\n            "action":{"id":rec.action.id.name,"data":dict(rec.action.data)},\n            "generated_tokens":int(getattr(rec,"generated_tokens",0) or 0),\n            "uncached_input_tokens":int(getattr(rec,"uncached_input_tokens",0) or 0),\n            "wallclock_seconds":float(getattr(rec,"wallclock_seconds",0.0) or 0.0),\n        })\n    return out\n\ndef install_advisory(path):\n    import inference.agent.tool_agent as tool_agent_module\n    spec=importlib.util.spec_from_file_location("r16_dual1024_advisory_runtime",path)\n    if spec is None or spec.loader is None:\n        raise RuntimeError("R16 advisory import spec failed")\n    mod=importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(mod)\n    mod.install(tool_agent_module)\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument("--spec",required=True)\n    ns=ap.parse_args()\n    cfg=json.loads(Path(ns.spec).read_text(encoding="utf-8"))\n    result={"status":"OPEN","game_id":cfg["game_id"],"tag":cfg.get("tag","")}\n    try:\n        import arc_agi, arcengine\n        from taaf.game import RunSession\n        from taaf.game_api import ArcadeSpec, GameAPI\n\n        install_advisory(cfg["advisory_path"])\n        gs=ArcadeSpec(\n            operation_mode=arc_agi.OperationMode.OFFLINE,\n            environments_dir=cfg["env_dir"],\n        )\n        game=GameAPI(env_name=cfg["game_id"],arcade_spec=gs)\n        session=RunSession(record_intermediate_states=False)\n        game.start_game(session)\n        for h in cfg.get("prefix_actions",[]):\n            game.execute_action(action_input(h,arcengine))\n        start=len(game.game_run.history)\n        before=int(game.current_state.levels_completed)\n        prefix_hash=state_hash(game.current_state.raw)\n        with open(Path(cfg["bundle_dir"])/"benchmark_initial.pkl","rb") as f:\n            bm=pickle.load(f)\n        solver=bm.solver\n        solver.job_dir=Path(cfg["job_dir"])\n        solver.max_runtime_s_per_game=float(cfg.get("branch_budget_s",300.0))\n        solver.analyzer_timeout=min(\n            float(getattr(solver,"analyzer_timeout",300.0) or 300.0),\n            float(cfg.get("branch_budget_s",300.0)),\n        )\n        solver.concurrency=1\n        solver.max_actions_per_game=None\n        solver.save_request_logs=False\n        solver.start_local_server=False\n        solver._setup()\n        try:\n            solver._play_one(game,0,0,None)\n        finally:\n            solver._teardown()\n\n        run=game.game_run\n        if run.final_score is None:\n            game.finish_game()\n        result.update({\n            "status":"AFFIRMED",\n            "prefix_hash":prefix_hash,\n            "levels_before":before,\n            "levels_completed":int(run.levels_completed),\n            "transitioned":int(run.levels_completed)>before,\n            "final_hash":state_hash(game.current_state.raw),\n            "branch_actions":hist_json(run,start),\n            "actions_per_level":list(run.actions_per_level),\n            "state":run.state,\n            "score_audit_only":run.final_score,\n            "solver_note":run.solver_note,\n        })\n        session.close()\n    except Exception as exc:\n        result["status"]="REJECTED"\n        result["error"]=repr(exc)\n        result["traceback"]=traceback.format_exc()\n    Path(cfg["output"]).write_text(\n        json.dumps(result,indent=2,sort_keys=True)+"\\n",\n        encoding="utf-8",\n    )\n    print(json.dumps({k:v for k,v in result.items() if k not in ("branch_actions","traceback")},sort_keys=True))\n    if result["status"]!="AFFIRMED":\n        raise SystemExit(2)\n\nif __name__=="__main__":\n    main()\n',encoding="utf-8")
VERIFY_PATH.write_text('from __future__ import annotations\nimport argparse, hashlib, json, os, traceback\nfrom pathlib import Path\n\nos.environ["ONLY_RESET_LEVELS"]="true"\n\ndef norm(v):\n    if hasattr(v,"tolist"): return v.tolist()\n    if isinstance(v,(list,tuple)): return [norm(x) for x in v]\n    if isinstance(v,dict): return {str(k):norm(x) for k,x in v.items()}\n    if hasattr(v,"value"): return v.value\n    return v\n\ndef state_obj(x):\n    return {\n        "frame":norm(x.frame),\n        "state":getattr(x.state,"name",str(x.state)),\n        "levels_completed":int(x.levels_completed),\n        "win_levels":int(x.win_levels),\n        "available_actions":[int(a) for a in x.available_actions],\n    }\n\ndef state_hash(x):\n    return hashlib.sha256(\n        json.dumps(state_obj(x),sort_keys=True,separators=(",",":")).encode()\n    ).hexdigest()\ndef do_action(env,h,arcengine):\n    a=h["action"]\n    return env.step(\n        arcengine.GameAction[a["id"]],\n        data=a.get("data") or {},\n    )\n\ndef replay(env,actions,arcengine,trace=False):\n    obs=env.observation_space\n    hashes=[]\n    for i,h in enumerate(actions):\n        obs=do_action(env,h,arcengine)\n        if obs is None:\n            raise RuntimeError(f"env.step returned None at {i}")\n        if trace:\n            hashes.append(state_hash(obs))\n    return obs,hashes\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument("--spec",required=True)\n    ns=ap.parse_args()\n    cfg=json.loads(Path(ns.spec).read_text(encoding="utf-8"))\n    out={"status":"OPEN","game_id":cfg["game_id"],"tag":cfg.get("tag","")}\n    try:\n        import arc_agi, arcengine\n        arcade=arc_agi.Arcade(\n            operation_mode=arc_agi.OperationMode.OFFLINE,\n            environments_dir=cfg["env_dir"],\n        )\n        scorecard=arcade.create_scorecard()\n        env=arcade.make(cfg["game_id"],scorecard_id=scorecard)\n        pref,_=replay(env,cfg.get("prefix_actions",[]),arcengine)\n        out["prefix_hash"]=state_hash(pref)\n        out["prefix_levels"]=int(pref.levels_completed)\n        before=int(pref.levels_completed)\n\n        contam=cfg.get("contamination_actions",[])\n        if contam:\n            _,_ = replay(env,contam,arcengine)\n        if cfg.get("reset_before_branch"):\n            reset=env.step(arcengine.GameAction.RESET,data={})\n            if reset is None:\n                raise RuntimeError("RESET returned None")\n            out["reset_hash"]=state_hash(reset)\n            out["reset_levels"]=int(reset.levels_completed)\n\n        final,trace=replay(env,cfg.get("branch_actions",[]),arcengine,trace=True)\n        out["branch_trace"]=trace\n        out["final_hash"]=state_hash(final)\n        out["final_levels"]=int(final.levels_completed)\n        out["transitioned"]=int(final.levels_completed)>before\n        out["status"]="AFFIRMED"\n        try:\n            arcade.close_scorecard(scorecard)\n        except Exception:\n            pass\n    except Exception as exc:\n        out["status"]="REJECTED"\n        out["error"]=repr(exc)\n        out["traceback"]=traceback.format_exc()\n\n    Path(cfg["output"]).write_text(\n        json.dumps(out,indent=2,sort_keys=True)+"\\n",\n        encoding="utf-8",\n    )\n    print(json.dumps({k:v for k,v in out.items() if k not in ("branch_trace","traceback")},sort_keys=True))\n    if out["status"]!="AFFIRMED":\n        raise SystemExit(2)\n\nif __name__=="__main__":\n    main()\n',encoding="utf-8")

ENV_DIR=str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent/"environment_files")
os.environ["R16_SCOUT_TOKENS"]="1024"
os.environ["R16_SCOUT_TIMEOUT_S"]="180"
print("R16_PAYLOAD",R16_PAYLOAD_SHA,"games",R16_GAMES,flush=True)


In [ ]:
def _branch_spec(game):
    row=R16_PAYLOAD[game]
    out=R16_DIR/f"{game}_live.json"
    spec={
        "tag":"kaggle-r16-residual6",
        "bundle_dir":str(BUNDLE_DIR),
        "env_dir":ENV_DIR,
        "game_id":row["game_id"],
        "prefix_actions":row["prefix_actions"],
        "branch_budget_s":300.0,
        "job_dir":str(R16_DIR/f"{game}_job"),
        "output":str(out),
        "advisory_path":str(ADV_PATH),
    }
    sp=R16_DIR/f"{game}_live_spec.json"
    sp.write_text(json.dumps(spec),encoding="utf-8")
    return sp,out

def _run_process(argv,timeout_s,log_path):
    started=time.time()
    cp=subprocess.run(
        argv,capture_output=True,text=True,timeout=timeout_s,
        env=os.environ.copy(),
    )
    Path(log_path).write_text(
        (cp.stdout or "")+"\n---STDERR---\n"+(cp.stderr or ""),
        encoding="utf-8",
    )
    return cp,time.time()-started

def _live_one(game):
    sp,out=_branch_spec(game)
    cp,elapsed=_run_process(
        [sys.executable,str(BRANCH_PATH),"--spec",str(sp)],
        420,R16_DIR/f"{game}_live.log",
    )
    if out.exists():
        obj=json.loads(out.read_text(encoding="utf-8"))
    else:
        obj={"status":"REJECTED","error":f"worker rc={cp.returncode} no output"}
    obj["_rc"]=cp.returncode
    obj["_elapsed_s"]=elapsed
    return game,obj

def _verify_one(game,live,commit):
    row=R16_PAYLOAD[game]
    branch_actions=list(live.get("branch_actions") or [])
    tag="commit" if commit else "clean"
    out=R16_DIR/f"{game}_{tag}.json"
    spec={
        "env_dir":ENV_DIR,
        "game_id":row["game_id"],
        "tag":tag,
        "prefix_actions":row["prefix_actions"],
        "branch_actions":branch_actions,
        "output":str(out),
    }
    if commit:
        spec["contamination_actions"]=row["contamination_actions"]
        spec["reset_before_branch"]=True
    sp=R16_DIR/f"{game}_{tag}_spec.json"
    sp.write_text(json.dumps(spec),encoding="utf-8")
    cp,elapsed=_run_process(
        [sys.executable,str(VERIFY_PATH),"--spec",str(sp)],
        180,R16_DIR/f"{game}_{tag}.log",
    )
    if out.exists():
        obj=json.loads(out.read_text(encoding="utf-8"))
    else:
        obj={"status":"REJECTED","error":f"verifier rc={cp.returncode} no output"}
    obj["_rc"]=cp.returncode
    obj["_elapsed_s"]=elapsed
    return game,tag,obj

def _teardown():
    for command in json.loads((BUNDLE_DIR/"teardown_commands.json").read_text()):
        print(f"taaf.kaggle: teardown command: {command}",flush=True)
        subprocess.run(
            command,shell=True,check=False,cwd=WORKING_DIR,
            env=_command_env(),timeout=30.0,
        )

started=time.time()
live_results={}
verify_results={}
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=6) as ex:
        futs=[ex.submit(_live_one,g) for g in R16_GAMES]
        for fut in concurrent.futures.as_completed(futs):
            g,obj=fut.result()
            live_results[g]=obj
            print("R16_LIVE",g,obj.get("status"),"gain",
                  int(obj.get("levels_completed",R16_PAYLOAD[g]["depth"]))-R16_PAYLOAD[g]["depth"],
                  "elapsed",round(obj.get("_elapsed_s",0),1),flush=True)

    jobs=[]
    with concurrent.futures.ThreadPoolExecutor(max_workers=12) as ex:
        for g in R16_GAMES:
            live=live_results[g]
            if live.get("status")=="AFFIRMED" and live.get("branch_actions"):
                jobs.append(ex.submit(_verify_one,g,live,False))
                jobs.append(ex.submit(_verify_one,g,live,True))
        for fut in concurrent.futures.as_completed(jobs):
            g,tag,obj=fut.result()
            verify_results[(g,tag)]=obj

    rows=[]
    proof_mismatches=0
    rescued=[]
    for g in R16_GAMES:
        row=R16_PAYLOAD[g]
        depth=int(row["depth"])
        live=live_results.get(g,{})
        clean=verify_results.get((g,"clean"),{})
        commit=verify_results.get((g,"commit"),{})
        gain=int(live.get("levels_completed",depth))-depth
        proof=(
            live.get("status")=="AFFIRMED"
            and clean.get("status")=="AFFIRMED"
            and commit.get("status")=="AFFIRMED"
            and gain>0
            and live.get("final_hash")==clean.get("final_hash")==commit.get("final_hash")
            and clean.get("branch_trace")==commit.get("branch_trace")
            and int(commit.get("reset_levels",-1))==depth
        )
        if gain>0 and not proof:
            proof_mismatches+=1
        if proof:
            rescued.append(g)
        rows.append({
            "game":g,"depth":depth,
            "live_status":live.get("status"),
            "live_levels":live.get("levels_completed"),
            "live_gain":gain,
            "live_actions":len(live.get("branch_actions") or []),
            "proof":proof,
            "score15_audit":row.get("score15_audit"),
            "score_live_audit":live.get("score_audit_only"),
            "solver_note":live.get("solver_note"),
            "live_error":live.get("error"),
            "clean_error":clean.get("error"),
            "commit_error":commit.get("error"),
        })

    priority={"ft09","ar25","sc25"}
    if proof_mismatches:
        status="REJECTED_PROOF_MISMATCH"
    elif len(rescued)>=3 and any(g in priority for g in rescued):
        status="AFFIRMED_FOR_FULL_PUBLIC25_GATE"
    elif len(rescued)>=1:
        status="BECOMING_REPEAT_RESIDUAL_ONLY"
    else:
        status="REJECTED_FOR_R16_PROMOTION"

    cert={
        "status":status,
        "carrier":"R16_DUAL1024_ISO300_KAGGLE_RESIDUAL6",
        "payload_sha256":R16_PAYLOAD_SHA,
        "branch_budget_s":300,
        "scout_count":2,
        "scout_output_tokens":1024,
        "generation_used_r12_trace":False,
        "admission_consumed_score":False,
        "competition_submission":False,
        "fruit_contract":{
            "promote":"3/6 certified rescues incl one of ft09/ar25/sc25 and 0 proof mismatches",
            "repeat":"1-2/6 certified rescues and 0 proof mismatches",
            "reject":"0/6 or any proof mismatch",
        },
        "rescued":rescued,
        "rescue_count":len(rescued),
        "proof_mismatches":proof_mismatches,
        "elapsed_s":time.time()-started,
        "rows":rows,
    }
    raw=json.dumps(cert,indent=2,sort_keys=True)+"\n"
    cert_path=WORKING_DIR/"R16_KAGGLE_RESIDUAL6_CERT.json"
    cert_path.write_text(raw,encoding="utf-8")
    print("R16_CERT_SHA256",hashlib.sha256(raw.encode()).hexdigest(),flush=True)
    print(json.dumps(cert,indent=2,sort_keys=True),flush=True)

    import pandas as pd
    pd.DataFrame([["1_0","1",True,1]],
        columns=["row_id","game_id","end_of_game","score"]
    ).to_parquet(WORKING_DIR/"submission.parquet",index=False)
finally:
    _teardown()
